In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
import torch.optim as optim

import pandas as pd
from PIL import Image
from torchvision import transforms

from tqdm import tqdm


In [2]:
from torch.utils.data import Dataset

LABEL2ID = {
    "NORMAL": 0,
    "PNEUMONIA": 1,
    "EFFUSION": 2,
    "PNEUMOTHORAX": 3,
    "CARDIOMEGALY": 4,
    "EDEMA": 5,
    "OTHER": 6
}

NUM_CLASSES = len(LABEL2ID)


In [3]:
class ImageOnlyDataset(Dataset):
    def __init__(self, csv_file, transform):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)

        label = torch.tensor(LABEL2ID[row["label"]], dtype=torch.long)

        return image, label


In [4]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [5]:
CSV_PATH = r"D:\multimodal-clinical-ai\multimodal-clinical-ai\data\metadata\patient_index.csv"

dataset = ImageOnlyDataset(CSV_PATH, image_transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
# val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)


print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))


Train samples: 24498
Val samples: 6125


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(device)


c:\Users\KRISH\miniconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\KRISH\miniconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [8]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


In [9]:
def validate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [10]:
EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_acc = validate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Train Acc : {train_acc:.4f}")
    print(f"  Val Acc   : {val_acc:.4f}")


100%|██████████| 766/766 [48:45<00:00,  3.82s/it]    


Epoch 1/5
  Train Loss: 1.5648
  Train Acc : 0.3603
  Val Acc   : 0.3709


100%|██████████| 766/766 [08:43<00:00,  1.46it/s]


Epoch 2/5
  Train Loss: 1.4119
  Train Acc : 0.4271
  Val Acc   : 0.3811


100%|██████████| 766/766 [09:23<00:00,  1.36it/s]


Epoch 3/5
  Train Loss: 1.2324
  Train Acc : 0.5095
  Val Acc   : 0.3734


100%|██████████| 766/766 [04:28<00:00,  2.86it/s]


Epoch 4/5
  Train Loss: 0.8665
  Train Acc : 0.6679
  Val Acc   : 0.3296


100%|██████████| 766/766 [06:17<00:00,  2.03it/s]


Epoch 5/5
  Train Loss: 0.3773
  Train Acc : 0.8718
  Val Acc   : 0.3505


In [11]:
torch.save(
    {
        "model_state": model.state_dict(),
        "label_map": LABEL2ID
    },
    r"D:\multimodal-clinical-ai\multimodal-clinical-ai\checkpoints/image_only_baseline.pt"
)

print("✅ Image-only baseline saved.")


✅ Image-only baseline saved.
